In [39]:
import pandas as pd, geopandas as gpd, folium
from IPython.display import HTML
pd.set_option("display.float_format", "{:.4f}".format)

In [40]:
ABT = gpd.read_file(
    "../../../../Data/Final_dataset/ABT/ABT.gpkg",
    layer="subdivisions"
)

npa_raw = gpd.read_file(
    "../../../../Data/Original_dataset/original.gdb",
    layer="QOL_NPA_2020_final_projected"
)

In [41]:
ABT_proj = ABT[ABT["year"].between(1990, 2023)].copy()
npa_proj = npa_raw.to_crs(ABT_proj.crs)

In [42]:
ABT_proj["subd_area"] = ABT_proj.geometry.area

In [43]:
abt_npa_intersections = gpd.overlay(
    ABT_proj[["subd_id", "geometry"]],
    npa_proj[["NPA_ID", "geometry"]],
    how="intersection"
)

abt_npa_intersections["intersect_area"] = (
    abt_npa_intersections.geometry.area
)

In [44]:
npa_area = (
    abt_npa_intersections
    .groupby(["subd_id", "NPA_ID"])["intersect_area"]
    .sum()
    .reset_index()
)

In [46]:
npa_count = (
    npa_area
    .groupby("subd_id")["NPA_ID"]
    .nunique()
    .reset_index(name="npa_count")
)

In [49]:
npa_id_list = (
    npa_area
    .groupby("subd_id")["NPA_ID"]
    .apply(lambda x: sorted(x.unique().tolist()))
    .reset_index(name="npa_id_list")
)

In [50]:
npa_area = npa_area.merge(
    ABT_proj[["subd_id", "subd_area"]],
    on="subd_id",
    how="left"
)

npa_area["share"] = (
    npa_area["intersect_area"] / npa_area["subd_area"]
)

In [51]:
npa_area = npa_area.sort_values(
    ["subd_id", "share"],
    ascending=[True, False]
)

npa_area["rank"] = (
    npa_area
    .groupby("subd_id")
    .cumcount()
    + 1
)

In [52]:
npa_area_top = npa_area[npa_area["rank"] <= 6].copy()

In [53]:
npa_share_wide = (
    npa_area_top
    .pivot_table(
        index="subd_id",
        columns="rank",
        values="share",
        fill_value=0
    )
)

npa_share_wide.columns = [
    f"share_npa_{int(c)}" for c in npa_share_wide.columns
]

npa_share_wide = npa_share_wide.reset_index()

In [55]:
ABT_proj = (
    ABT_proj
    .merge(npa_share_wide, on="subd_id", how="left")
    .merge(npa_count, on="subd_id", how="left")
    .merge(npa_id_list, on="subd_id", how="left")
)

share_cols = [f"share_npa_{k}" for k in range(1, 7)]
ABT_proj[share_cols] = ABT_proj[share_cols].fillna(0)
ABT_proj["npa_count"] = ABT_proj["npa_count"].fillna(0).astype(int)
ABT_proj["npa_id_list"] = ABT_proj["npa_id_list"].fillna("").astype(object)

In [15]:
ABT_proj["share_outside_npa"] = (
    1 - ABT_proj[share_cols].sum(axis=1)
).clip(lower=0)

In [16]:
ABT_proj["share_sum_check"] = (
    ABT_proj[share_cols + ["share_outside_npa"]]
    .sum(axis=1)
)

ABT_proj["share_sum_check"].describe()

assert ABT_proj["share_sum_check"].between(0.999, 1.001).all(), (
    "Area shares do not sum to 1 — check geometry validity"
)

In [17]:
npa_count_freq = (
    ABT_proj["npa_count"]
    .value_counts()
    .sort_index()
    .rename("count")
    .reset_index()
    .rename(columns={"index": "npa_count"})
)

npa_count_freq

,npa_count,count
0,1,4646
1,2,958
2,3,203
3,4,30
4,5,6
5,6,1


In [18]:
# Share columns
share_cols = [f"share_npa_{k}" for k in range(1, 7)] + ["share_outside_npa"]

# Create percentage versions
for col in share_cols:
    ABT_proj[f"{col}_pct"] = ABT_proj[col] * 100


In [20]:
pct_cols = [
    "share_npa_1_pct",
    "share_npa_2_pct",
    "share_npa_3_pct",
    "share_npa_4_pct",
    "share_npa_5_pct",
    "share_npa_6_pct",
    "share_outside_npa_pct"
]

desc_stats = (
    ABT_proj[pct_cols]
    .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    .T
)

desc_stats

,count,mean,std,min,5%,25%,50%,75%,95%,max
share_npa_1_pct,5844.0000,99.2788,4.7865,15.1833,99.2894,100.0000,100.0000,100.0000,100.0000,100.0000
share_npa_2_pct,5844.0000,0.6175,4.1389,0.0000,0.0000,0.0000,0.0000,0.0000,0.4724,49.9951
share_npa_3_pct,5844.0000,0.0167,0.4654,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,23.0708
share_npa_4_pct,5844.0000,0.0011,0.0726,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,5.5233
share_npa_5_pct,5844.0000,0.0000,0.0004,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0276
share_npa_6_pct,5844.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0001
share_outside_npa_pct,5844.0000,0.0859,2.0888,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,84.8167


**Subdivisions with Low Dominant NPA Overlap (share < 80%)**

In [21]:
threshold = 80

split_subs = ABT_proj[
    ABT_proj["share_npa_1_pct"] < threshold
    ].copy()

len(split_subs)

77

In [22]:
split_subs_wgs = split_subs.to_crs(epsg=4326)
npa_wgs = npa_proj.to_crs(epsg=4326)

In [23]:
center = split_subs_wgs.geometry.unary_union.centroid
m = folium.Map(
    location=[center.y, center.x],
    zoom_start=11,
    tiles="CartoDB positron"
)

C:\Users\erfan\AppData\Local\Temp\ipykernel_11180\4288685633.py:1: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = split_subs_wgs.geometry.unary_union.centroid


In [24]:
folium.GeoJson(
    npa_wgs,
    name="NPAs",
    style_function=lambda x: {
        "fillColor": "#d9d9d9",
        "color": "#666666",
        "weight": 1,
        "fillOpacity": 0.25,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["NPA_ID"],
        aliases=["NPA ID:"]
    )
).add_to(m)

In [25]:
def style_split(feature):
    return {
        "fillColor": "#e41a1c",
        "color": "#e41a1c",
        "weight": 2,
        "fillOpacity": 0.7,
    }


folium.GeoJson(
    split_subs_wgs,
    name="Split / Weak-Dominant Subdivisions",
    style_function=style_split,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "subd_id",
            "npa_count",
            "share_npa_1_pct",
            "share_npa_2_pct",
            "share_outside_npa_pct"
        ],
        aliases=[
            "Subdivision ID:",
            "NPA count:",
            "Top NPA share (%):",
            "Second NPA share (%):",
            "Outside NPA (%):"
        ],
        localize=True
    )
).add_to(m)

In [26]:
folium.LayerControl(collapsed=False).add_to(m)
m

**Create dominant NPA ID + dominance share**

In [27]:
# Identify dominant NPA per subdivision
dominant_npa = (
    npa_area
    .sort_values(["subd_id", "share"], ascending=[True, False])
    .groupby("subd_id", as_index=False)
    .first()[["subd_id", "NPA_ID", "share"]]
    .rename(columns={
        "NPA_ID": "dominant_npa_id",
        "share": "dominant_npa_share"
    })
)

# Ensure integer type for ID
dominant_npa["dominant_npa_id"] = dominant_npa["dominant_npa_id"].astype(int)

ABT_proj = ABT_proj.merge(
    dominant_npa,
    on="subd_id",
    how="left"
)

In [33]:
ABT_proj_npa_ready = ABT_proj[["year", "dominant_npa_id",'HAC_dist', 'BAD', 'SHD', 'int_den025','nd_deg025', 'int_den05', 'nd_deg05', 'int_den075', 'nd_deg075',
       'int_den1', 'nd_deg1', 'AI', 'PROX', 'ENN_MN', 'ED', 'SHAPE_MN',
       'FRAC_MN', 'ENN_inv', 'ED_inv', 'SHAPE_inv', 'FRAC_inv', 'AI_norm',
       'PROX_norm', 'ENN_inv_norm', 'ED_inv_norm', 'SHAPE_inv_norm',
       'FRAC_inv_norm', 'COMPACTNESS_SUM', 'BAD_ctx_025', 'BAD_ctx_050',
       'groceries_ws', 'transit_ws', 'FAR']]

In [37]:
NPA_panel = (
    ABT_proj_npa_ready
    .groupby(["year", "dominant_npa_id"], as_index=False)
    .mean(numeric_only=True)
    # .reset_index()
)

In [78]:
NPA_panel

,year,dominant_npa_id,HAC_dist,BAD,SHD,int_den025,nd_deg025,int_den05,nd_deg05,int_den075,...,ENN_inv_norm,ED_inv_norm,SHAPE_inv_norm,FRAC_inv_norm,COMPACTNESS_SUM,BAD_ctx_025,BAD_ctx_050,groceries_ws,transit_ws,FAR
0,1990.0000,3,0.4750,0.2915,0.0000,0.1655,2.4986,0.1949,2.8165,0.1777,...,0.0470,0.0021,0.8286,0.8322,0.0312,0.1740,0.1960,87.5700,44.5000,0.4208
1,1990.0000,7,0.3200,0.2990,0.8900,0.1144,2.3922,0.0743,2.4000,0.1010,...,0.0223,0.0076,0.7887,0.8413,0.0497,0.1870,0.1940,94.2800,37.0000,0.5805
2,1990.0000,11,0.9500,0.1710,0.0800,0.1107,2.0000,0.1330,2.3066,0.1374,...,0.0541,0.0033,0.7356,0.7561,0.0312,0.1440,0.1420,62.5300,31.0000,0.2797
3,1990.0000,13,1.9200,0.2530,0.3800,0.1183,2.1429,0.2066,2.4409,0.1671,...,0.0150,0.0035,0.9140,0.9016,0.0234,0.1930,0.1810,97.1400,50.0000,0.2552
4,1990.0000,17,4.0700,0.1520,0.4300,0.1292,2.2667,0.1268,2.2313,0.1259,...,0.0445,0.0046,0.7088,0.7357,0.0296,0.1190,0.1030,96.1700,39.0000,0.2556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3993,2023.0000,456,6.4300,0.3970,0.0000,0.1277,2.2778,0.1427,2.4516,0.1200,...,0.0754,0.0008,0.8780,0.8646,0.0391,0.1330,0.1220,86.1800,0.0000,0.5280
3994,2023.0000,462,3.0950,0.2470,0.2700,0.3166,2.4104,0.2886,2.7165,0.2015,...,0.0787,0.0021,0.9071,0.8849,0.0403,0.0680,0.0920,55.0200,0.0000,0.3836
3995,2023.0000,467,1.7600,0.1840,0.5800,0.1515,2.4516,0.1029,2.5641,0.0713,...,0.0507,0.0042,0.5097,0.6144,0.0322,0.0690,0.0840,3.1500,0.0000,0.3964
3996,2023.0000,471,0.4300,0.2080,0.4700,0.3293,2.3585,0.2167,2.4695,0.1623,...,0.0555,0.0037,0.8391,0.8380,0.0344,0.1320,0.1050,85.6700,0.0000,0.4199


In [77]:
NPA_panel.to_file("../../../Data/Final_dataset/ABT/NPA_panel.gpkg", layer="NPA_panel")

AttributeError: 'DataFrame' object has no attribute 'to_file'

In [58]:
panel = NPA_panel.copy()

panel = panel.sort_values(["dominant_npa_id", "year"])
panel = panel.set_index(["dominant_npa_id", "year"])

panel.index.is_monotonic_increasing
panel.isna().mean().sort_values(ascending=False).head()

HAC_dist       0.0000
ENN_inv        0.0000
transit_ws     0.0000
groceries_ws   0.0000
BAD_ctx_050    0.0000
dtype: float64

In [59]:
panel.reset_index().groupby("dominant_npa_id")["year"].count().describe()


count   443.0000
mean      9.0248
std       5.8109
min       1.0000
25%       5.0000
50%       8.0000
75%      12.5000
max      29.0000
Name: year, dtype: float64

In [62]:
from arch.unitroot import ADF, KPSS

mean_series = (
    panel
    .groupby("year")["COMPACTNESS_SUM"]
    .mean()
)

ADF(mean_series).summary()
KPSS(mean_series).summary()

C:\Users\erfan\AppData\Local\Temp\ipykernel_11180\1593393097.py:10: DeprecationWarning: Lag selection has changed to use a data-dependent method. To use the old method that only depends on time, set lags=-1
  KPSS(mean_series).summary()


Test Statistic,0.812
P-value,0.007
Lags,3


In [67]:
from linearmodels.panel import PanelOLS
import statsmodels.api as sm
import pandas as pd

y = panel["HAC_dist"]

# Extract year from index and wrap as DataFrame
X = pd.DataFrame(
    {"year": panel.index.get_level_values("year")},
    index=panel.index
)

X = sm.add_constant(X)

fe = PanelOLS(
    y,
    X,
    entity_effects=True
).fit(
    cov_type="clustered",
    cluster_entity=True
)

print(fe.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:               HAC_dist   R-squared:                        0.0049
Estimator:                   PanelOLS   R-squared (Between):             -0.0002
No. Observations:                3998   R-squared (Within):               0.0049
Date:                Thu, Feb 05 2026   R-squared (Overall):             -0.0002
Time:                        22:02:28   Log-likelihood                   -1694.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      17.517
Entities:                         443   P-value                           0.0000
Avg Obs:                       9.0248   Distribution:                  F(1,3554)
Min Obs:                       1.0000                                           
Max Obs:                       29.000   F-statistic (robust):             13.340
                            

In [69]:
from linearmodels.panel import PanelOLS
import statsmodels.api as sm
import pandas as pd

def fe_trend(panel, yvar):
    y = panel[yvar]
    X = pd.DataFrame(
        {"year": panel.index.get_level_values("year")},
        index=panel.index
    )
    X = sm.add_constant(X)

    res = PanelOLS(
        y, X,
        entity_effects=True
    ).fit(
        cov_type="clustered",
        cluster_entity=True
    )
    return res

res_hac  = fe_trend(panel, "HAC_dist")
res_comp = fe_trend(panel, "COMPACTNESS_SUM")
res_far  = fe_trend(panel, "FAR")

print(res_comp.summary)
print(res_far.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:        COMPACTNESS_SUM   R-squared:                        0.0063
Estimator:                   PanelOLS   R-squared (Between):              0.0129
No. Observations:                3998   R-squared (Within):               0.0063
Date:                Thu, Feb 05 2026   R-squared (Overall):              0.0114
Time:                        22:04:47   Log-likelihood                    3941.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      22.634
Entities:                         443   P-value                           0.0000
Avg Obs:                       9.0248   Distribution:                  F(1,3554)
Min Obs:                       1.0000                                           
Max Obs:                       29.000   F-statistic (robust):             19.553
                            

In [70]:
panel_std = panel.copy()
for v in ["HAC_dist", "COMPACTNESS_SUM", "FAR"]:
    panel_std[v] = (
        panel_std[v] - panel_std[v].mean()
    ) / panel_std[v].std()

res_std = fe_trend(panel_std, "COMPACTNESS_SUM")
print(res_std.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:        COMPACTNESS_SUM   R-squared:                        0.0063
Estimator:                   PanelOLS   R-squared (Between):              0.0129
No. Observations:                3998   R-squared (Within):               0.0063
Date:                Thu, Feb 05 2026   R-squared (Overall):              0.0114
Time:                        22:04:58   Log-likelihood                   -5188.0
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      22.634
Entities:                         443   P-value                           0.0000
Avg Obs:                       9.0248   Distribution:                  F(1,3554)
Min Obs:                       1.0000                                           
Max Obs:                       29.000   F-statistic (robust):             19.553
                            

In [71]:
def fe_tw(panel, yvar):
    y = panel[yvar]
    X = pd.DataFrame(
        {"year": panel.index.get_level_values("year")},
        index=panel.index
    )
    X = sm.add_constant(X)

    res = PanelOLS(
        y, X,
        entity_effects=True,
        time_effects=True
    ).fit(
        cov_type="clustered",
        cluster_entity=True
    )
    return res

res_hac_tw = fe_tw(panel, "HAC_dist")
print(res_hac_tw.summary)


AbsorbingEffectError: 
The model cannot be estimated. The included effects have fully absorbed
one or more of the variables. This occurs when one or more of the dependent
variable is perfectly explained using the effects included in the model.

The following variables or variable combinations have been fully absorbed
or have become perfectly collinear after effects are removed:

          const, year

Set drop_absorbed=True to automatically drop absorbed variables.


In [72]:
mean_ts = (
    panel
    .reset_index()
    .groupby("year")
    .mean(numeric_only=True)
)

import ruptures as rpt
import numpy as np

y = mean_ts["HAC_dist"].values
y_std = (y - y.mean()) / y.std()

algo = rpt.Pelt(model="rbf").fit(y_std)
breaks = algo.predict(pen=3 * np.log(len(y_std)))

break_years = mean_ts.index.values[np.array(breaks)[:-1]]
break_years



array([], dtype=float64)

In [74]:
panel_dyn = panel.copy()
panel_dyn["HAC_L1"] = (
    panel_dyn
    .groupby(level=0)["HAC_dist"]
    .shift(1)
)

panel_dyn = panel_dyn.dropna(subset=["HAC_L1"])

y = panel_dyn["HAC_dist"]

X = panel_dyn[["HAC_L1"]].copy()
X["year"] = panel_dyn.index.get_level_values("year")
X = sm.add_constant(X)

dyn_fe = PanelOLS(
    y, X,
    entity_effects=True
).fit(
    cov_type="clustered",
    cluster_entity=True
)

print(dyn_fe.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:               HAC_dist   R-squared:                        0.0049
Estimator:                   PanelOLS   R-squared (Between):             -0.0728
No. Observations:                3555   R-squared (Within):               0.0049
Date:                Thu, Feb 05 2026   R-squared (Overall):             -0.0719
Time:                        22:05:58   Log-likelihood                   -1533.7
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      7.6460
Entities:                         419   P-value                           0.0005
Avg Obs:                       8.4845   Distribution:                  F(2,3134)
Min Obs:                       1.0000                                           
Max Obs:                       28.000   F-statistic (robust):             5.1520
                            

In [75]:
X = panel[["int_den05", "nd_deg05", "transit_ws"]].copy()
X["year"] = panel.index.get_level_values("year")
X = sm.add_constant(X)

fe_controls = PanelOLS(
    panel["HAC_dist"],
    X,
    entity_effects=True
).fit(
    cov_type="clustered",
    cluster_entity=True
)

print(fe_controls.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:               HAC_dist   R-squared:                        0.0696
Estimator:                   PanelOLS   R-squared (Between):              0.0836
No. Observations:                3998   R-squared (Within):               0.0696
Date:                Thu, Feb 05 2026   R-squared (Overall):              0.0947
Time:                        22:06:22   Log-likelihood                   -1560.4
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      66.360
Entities:                         443   P-value                           0.0000
Avg Obs:                       9.0248   Distribution:                  F(4,3551)
Min Obs:                       1.0000                                           
Max Obs:                       29.000   F-statistic (robust):             12.773
                            

In [32]:
!jupyter nbconvert --to html --no-input EDA.ipynb --output ../../../../output/Notebook_Outputs/spatio_temporal/EDA.html

[NbConvertApp] Converting notebook EDA.ipynb to html
[NbConvertApp] Writing 5804996 bytes to ..\..\..\..\output\Notebook_Outputs\spatio_temporal\EDA.html
